# DSN AI Bootcamp 2026
## Multilingual Topic Classification and Headline Generation

### 1. Project Overview
### 2. Imports and Setup
### 3. Load the Dataset
### 4. Exploratory Data Analysis
### 5. Classical Baseline
### 6. Zero-Shot LLM Baseline
### 7. Few-Shot Prompting
### 8. LoRA Fine-Tuning
### 9. Model Evaluation
### 10. Error Analysis
### 11. Baseline vs Final Model
### 12. Generate Test Predictions
### 13. Create submission.csv
### 14. Conclusion
### 15. Model Card

In [ ]:
!pip install -q scikit-learn

In [ ]:
import sklearn
print("scikit-learn version:", sklearn.__version__)

In [ ]:
import sklearn
print("sklearn location:", sklearn.__file__)
print("sklearn version:", getattr(sklearn, "__version__", "No version found"))

In [ ]:
!pip uninstall -y scikit-learn sklearn
!pip install --no-cache-dir --force-reinstall scikit-learn

In [ ]:
!pip uninstall -y scikit-learn sklearn

In [ ]:
!pip install --no-cache-dir --force-reinstall scikit-learn==1.6.1

In [ ]:
import sklearn

print("scikit-learn version:", sklearn.__version__)

from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("All sklearn imports successful!")

In [ ]:
import os
import re
import json
import random
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Setup complete.")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
DATA_DIR = "/kaggle/input/competitions/dsn-bootcamp-hackathon-2026-llm-agent-track"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nFirst 5 training rows:")
display(train.head())

In [ ]:
print("Categories:")
print(train["category"].value_counts())

In [ ]:
print("Languages:")
print(train["lang"].value_counts())

In [ ]:
print("\nTest languages:")
print(test["lang"].value_counts())

In [ ]:
print(train["split"].value_counts(dropna=False))

In [ ]:
print("Missing values in training data:")
print(train.isnull().sum())

print("\nMissing values in test data:")
print(test.isnull().sum())

In [ ]:
for lang in train["lang"].unique():

    print("=" * 100)
    print("LANGUAGE:", lang)

    sample = train[
        train["lang"] == lang
    ].sample(
        2,
        random_state=42
    )

    for _, row in sample.iterrows():

        print("\nCATEGORY:", row["category"])
        print("HEADLINE:", row["headline"])
        print("TEXT:", row["text"][:1000])
        print()

In [ ]:
display(sample_submission.head(20))

In [ ]:
print("Sample submission shape:", sample_submission.shape)
print("\nFirst IDs:")
print(sample_submission["id"].head(20).tolist())

In [ ]:
print("Split distribution:")
print(train["split"].value_counts(dropna=False))

In [ ]:
print("Unique split values:")
print(train["split"].unique())

In [ ]:
from sklearn.model_selection import train_test_split

train_data, dev_data = train_test_split(
    train,
    test_size=0.15,
    random_state=42,
    stratify=train["category"]
)

print("Training set:", train_data.shape)
print("Validation set:", dev_data.shape)

In [ ]:
print("Training categories:")
print(train_data["category"].value_counts())

print("\nValidation categories:")
print(dev_data["category"].value_counts())

In [ ]:
pd.crosstab(
    dev_data["lang"],
    dev_data["category"]
)

In [ ]:
X_train = train_data["text"].fillna("")
y_train = train_data["category"]

X_dev = dev_data["text"].fillna("")
y_dev = dev_data["category"]

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_dev_tfidf = tfidf.transform(X_dev)

print("Training matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_dev_tfidf.shape)

In [ ]:
classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

classifier.fit(
    X_train_tfidf,
    y_train
)

print("Model trained successfully!")

In [ ]:
classical_predictions = classifier.predict(X_dev_tfidf)

print(classical_predictions[:20])

In [ ]:
classical_macro_f1 = f1_score(
    y_dev,
    classical_predictions,
    average="macro"
)

print(
    f"Classical Baseline Macro-F1: "
    f"{classical_macro_f1:.4f}"
)

In [ ]:
print(
    classification_report(
        y_dev,
        classical_predictions
    )
)

In [ ]:
dev_results = dev_data.copy()

dev_results["prediction"] = classical_predictions

In [ ]:
language_scores = {}

for language in sorted(dev_results["lang"].unique()):

    subset = dev_results[
        dev_results["lang"] == language
    ]

    score = f1_score(
        subset["category"],
        subset["prediction"],
        average="macro"
    )

    language_scores[language] = score

language_scores

In [ ]:
for language, score in language_scores.items():
    print(f"{language}: {score:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(
    y_dev,
    classical_predictions,
    labels=sorted(train["category"].unique())
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=sorted(train["category"].unique()),
    yticklabels=sorted(train["category"].unique())
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Classical Baseline Confusion Matrix")
plt.show()

In [ ]:
baseline_results = pd.DataFrame({
    "Model": ["TF-IDF + Logistic Regression"],
    "Macro-F1": [classical_macro_f1]
})

baseline_results

In [ ]:
def extractive_headline(text, max_words=20):
    text = str(text).strip()

    # Clean excessive whitespace
    text = re.sub(r"\s+", " ", text)

    # Split into sentences
    sentences = re.split(r"(?<=[.!?])\s+", text)

    # Remove empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]

    if not sentences:
        return text[:150]

    headline = sentences[0]

    # Limit length
    words = headline.split()

    if len(words) > max_words:
        headline = " ".join(words[:max_words])

    return headline

In [ ]:
dev_results["classical_headline"] = (
    dev_results["text"]
    .apply(extractive_headline)
)

In [ ]:
display(
    dev_results[
        ["lang", "category", "headline", "classical_headline"]
    ].head(10)
)

In [ ]:
!pip install -q rouge-score

In [ ]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False
)

In [ ]:
rouge_scores = []

for _, row in dev_results.iterrows():

    reference = str(row["headline"])
    prediction = str(row["classical_headline"])

    score = rouge.score(
        reference,
        prediction
    )

    rouge_scores.append(
        score["rougeL"].fmeasure
    )

classical_rouge_l = np.mean(rouge_scores)

print(
    f"Classical Headline ROUGE-L F1: "
    f"{classical_rouge_l:.4f}"
)

In [ ]:
dev_results["rougeL"] = rouge_scores

In [ ]:
for language in sorted(dev_results["lang"].unique()):

    score = dev_results.loc[
        dev_results["lang"] == language,
        "rougeL"
    ].mean()

    print(
        f"{language}: {score:.4f}"
    )

In [ ]:
worst_headlines = dev_results.sort_values(
    "rougeL"
).head(10)

display(
    worst_headlines[
        [
            "lang",
            "category",
            "headline",
            "classical_headline",
            "rougeL"
        ]
    ]
)

In [ ]:
baseline_results = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Lead-1 Extractive"
    ],
    "Macro-F1": [
        classical_macro_f1,
        np.nan
    ],
    "ROUGE-L": [
        np.nan,
        classical_rouge_l
    ]
})

baseline_results

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
import transformers

print("Transformers version:", transformers.__version__)

In [ ]:
import torch

print(
    "Available GPU memory:",
    torch.cuda.get_device_properties(0).total_memory / 1e9
    if torch.cuda.is_available()
    else "No GPU"
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)

In [ ]:
prompt = """You are a news topic classifier.

Classify the following news article into exactly one of these categories:
business, health, politics, religion, sports, entertainment, technology.

Article:
The government announced a new programme to support small businesses with access to funding and training.

Return only the category name.
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Model response:", response)

In [ ]:
LABELS = [
    "business",
    "health",
    "politics",
    "religion",
    "sports",
    "entertainment",
    "technology"
]

def extract_label(response):
    """
    Extract one of the valid topic labels from the model response.
    """
    response = response.lower().strip()

    for label in LABELS:
        if label in response:
            return label

    return "unknown"


def zero_shot_topic(text):
    prompt = f"""You are a multilingual news topic classifier.

Classify the news article into exactly ONE of these categories:

business
health
politics
religion
sports
entertainment
technology

The article may be written in Hausa, Igbo, Yoruba, or Nigerian Pidgin.

Return ONLY the category name.

News article:
{text}
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return extract_label(response)

In [ ]:
for i in range(5):
    row = dev_data.iloc[i]

    prediction = zero_shot_topic(row["text"])

    print("=" * 70)
    print("Language:", row["lang"])
    print("Actual:", row["category"])
    print("Predicted:", prediction)
    print("Headline:", row["headline"])

In [ ]:
zero_shot_predictions = []

start_time = time.time()

for i, text in enumerate(dev_data["text"].fillna("")):
    prediction = zero_shot_topic(text)
    zero_shot_predictions.append(prediction)

    if (i + 1) % 25 == 0:
        elapsed = time.time() - start_time
        print(f"Processed {i + 1}/{len(dev_data)} articles | "
              f"Time: {elapsed/60:.1f} min")

print("Finished!")

In [ ]:
zero_shot_f1 = f1_score(
    dev_data["category"],
    zero_shot_predictions,
    average="macro"
)

print(f"Zero-shot Macro-F1: {zero_shot_f1:.6f}")

In [ ]:
dev_results["zero_shot_prediction"] = zero_shot_predictions

In [ ]:
print("Zero-shot Macro-F1 by language")
print("=" * 50)

for language in sorted(dev_data["lang"].unique()):
    mask = dev_data["lang"] == language

    score = f1_score(
        dev_data.loc[mask, "category"],
        np.array(zero_shot_predictions)[mask],
        average="macro"
    )

    print(f"{language}: {score:.6f}")

In [ ]:
baseline_results = pd.DataFrame([
    {
        "Approach": "TF-IDF + Logistic Regression",
        "Task": "Topic Classification",
        "Macro-F1": classical_macro_f1,
        "ROUGE-L": np.nan
    },
    {
        "Approach": "Lead-1 Extractive",
        "Task": "Headline Generation",
        "Macro-F1": np.nan,
        "ROUGE-L": 0.238261
    },
    {
        "Approach": "Qwen2.5-0.5B Zero-Shot",
        "Task": "Topic Classification",
        "Macro-F1": zero_shot_f1,
        "ROUGE-L": np.nan
    }
])

baseline_results

In [ ]:
print(f"Zero-shot Macro-F1: {zero_shot_f1:.6f}")

In [ ]:
# One example for each category and language
few_shot_examples = {}

for language in sorted(train_data["lang"].unique()):
    language_data = train_data[train_data["lang"] == language]

    examples = []

    for category in LABELS:
        candidates = language_data[
            language_data["category"] == category
        ]

        if len(candidates) > 0:
            row = candidates.iloc[0]

            # Keeping examples short to control prompt length
            example_text = str(row["text"]).strip()
            example_text = re.sub(r"\s+", " ", example_text)

            if len(example_text) > 700:
                example_text = example_text[:700] + "..."

            examples.append({
                "text": example_text,
                "category": category
            })

    few_shot_examples[language] = examples

for language, examples in few_shot_examples.items():
    print(language, "->", len(examples), "examples")

In [ ]:
def few_shot_topic(text, language):
    examples = few_shot_examples.get(language, [])

    example_block = ""

    for i, example in enumerate(examples, 1):
        example_block += f"""
Example {i}
Article:
{example["text"]}

Topic:
{example["category"]}
"""

    prompt = f"""You are a multilingual news topic classifier.

The article is written in {language}.

Choose exactly ONE topic from:

business
health
politics
religion
sports
entertainment
technology

Study the examples and classify the new article.

{example_block}

Now classify this article:

{text}

Return ONLY the topic name.
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return extract_label(response)

In [ ]:
for i in range(5):
    row = dev_data.iloc[i]

    prediction = few_shot_topic(
        row["text"],
        row["lang"]
    )

    print("=" * 70)
    print("Language:", row["lang"])
    print("Actual:", row["category"])
    print("Predicted:", prediction)

In [ ]:
few_shot_predictions = []

start_time = time.time()

for i, (_, row) in enumerate(dev_data.iterrows()):
    prediction = few_shot_topic(
        row["text"],
        row["lang"]
    )

    few_shot_predictions.append(prediction)

    if (i + 1) % 25 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {i+1}/{len(dev_data)} | "
            f"Time: {elapsed/60:.1f} min"
        )

print("Few-shot classification complete.")

In [ ]:
few_shot_f1 = f1_score(
    dev_data["category"],
    few_shot_predictions,
    average="macro"
)

print(f"Few-shot Macro-F1: {few_shot_f1:.6f}")

In [ ]:
print("Few-shot Macro-F1 by language")
print("=" * 50)

few_shot_array = np.array(few_shot_predictions)

for language in sorted(dev_data["lang"].unique()):
    mask = dev_data["lang"].to_numpy() == language

    score = f1_score(
        dev_data.loc[mask, "category"],
        few_shot_array[mask],
        average="macro"
    )

    print(f"{language}: {score:.6f}")

In [ ]:
def generate_headline(text):
    prompt = f"""You are a professional multilingual news editor.

Write a concise and informative news headline for the article below.

Important instructions:
- Write the headline in the same language as the article.
- Preserve the main event or information.
- Do not invent facts.
- Do not add explanations.
- Return ONLY the headline.
- Keep it concise.

Article:
{text}
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    headline = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    headline = re.sub(r"\s+", " ", headline).strip()

    return headline

In [ ]:
for i in range(5):
    row = dev_data.iloc[i]

    generated = generate_headline(row["text"])

    print("=" * 70)
    print("Language:", row["lang"])
    print("Reference:", row["headline"])
    print("Generated:", generated)

In [ ]:
llm_headlines = []

start_time = time.time()

for i, text in enumerate(dev_data["text"].fillna("")):
    headline = generate_headline(text)
    llm_headlines.append(headline)

    if (i + 1) % 25 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {i+1}/{len(dev_data)} | "
            f"Time: {elapsed/60:.1f} min"
        )

print("Headline generation complete.")

In [ ]:
rouge_scores = []

for reference, prediction in zip(
    dev_data["headline"].fillna(""),
    llm_headlines
):
    score = rouge.score(
        str(reference),
        str(prediction)
    )

    rouge_scores.append(score["rougeL"].fmeasure)

llm_rouge_l = np.mean(rouge_scores)

print(f"Qwen Zero-Shot Headline ROUGE-L: {llm_rouge_l:.6f}")

In [ ]:
!pip install -q bert-score

In [ ]:
from bert_score import score as bert_score

In [ ]:
P, R, F1 = bert_score(
    llm_headlines,
    dev_data["headline"].fillna("").tolist(),
    model_type="bert-base-multilingual-cased",
    device="cuda",
    batch_size=16,
    verbose=True
)

llm_bertscore = F1.mean().item()

print(f"BERTScore F1: {llm_bertscore:.6f}")

In [ ]:
llm_genscore = (
    0.5 * llm_rouge_l +
    0.5 * llm_bertscore
)

print(f"Qwen GenScore: {llm_genscore:.6f}")

In [ ]:
lead_bertscore = bert_score(
    dev_results["classical_headline"].tolist(),
    dev_data["headline"].fillna("").tolist(),
    model_type="bert-base-multilingual-cased",
    device="cuda",
    batch_size=16,
    verbose=True
)[2].mean().item()

lead_genscore = (
    0.5 * 0.238261 +
    0.5 * lead_bertscore
)

print(f"Lead-1 GenScore: {lead_genscore:.6f}")

In [ ]:
results = pd.DataFrame([
    {
        "Approach": "TF-IDF + Logistic Regression",
        "Task": "Topic Classification",
        "Macro-F1": classical_macro_f1,
        "ROUGE-L": np.nan,
        "BERTScore": np.nan,
        "GenScore": np.nan
    },
    {
        "Approach": "Qwen2.5-0.5B Zero-Shot",
        "Task": "Topic Classification",
        "Macro-F1": zero_shot_f1,
        "ROUGE-L": np.nan,
        "BERTScore": np.nan,
        "GenScore": np.nan
    },
    {
        "Approach": "Qwen2.5-0.5B Few-Shot",
        "Task": "Topic Classification",
        "Macro-F1": few_shot_f1,
        "ROUGE-L": np.nan,
        "BERTScore": np.nan,
        "GenScore": np.nan
    },
    {
        "Approach": "Lead-1 Extractive",
        "Task": "Headline Generation",
        "Macro-F1": np.nan,
        "ROUGE-L": 0.238261,
        "BERTScore": lead_bertscore,
        "GenScore": lead_genscore
    },
    {
        "Approach": "Qwen2.5-0.5B Zero-Shot",
        "Task": "Headline Generation",
        "Macro-F1": np.nan,
        "ROUGE-L": llm_rouge_l,
        "BERTScore": llm_bertscore,
        "GenScore": llm_genscore
    }
])

results

In [ ]:
!pip install -q peft accelerate

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

print("PEFT imported successfully")

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none"
)

In [ ]:
!pip uninstall -y torchao

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import peft

print("PEFT:", peft.__version__)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)

In [ ]:
model_lora = get_peft_model(
    model,
    lora_config
)

model_lora.print_trainable_parameters()

In [ ]:
def create_topic_prompt(row):
    return f"""You are a multilingual news topic classifier.

The article is written in {row['lang']}.

Classify the article into exactly one of these categories:
business, health, politics, religion, sports, entertainment, technology.

Article:
{row['text']}

Topic:
{row['category']}"""

In [ ]:
print(create_topic_prompt(train_data.iloc[0]))

In [ ]:
from datasets import Dataset

train_topic_df = train_data[["text", "category", "lang"]].copy()
dev_topic_df = dev_data[["text", "category", "lang"]].copy()

train_topic_df["prompt"] = train_topic_df.apply(
    create_topic_prompt,
    axis=1
)

dev_topic_df["prompt"] = dev_topic_df.apply(
    create_topic_prompt,
    axis=1
)

train_dataset = Dataset.from_pandas(
    train_topic_df[["prompt"]],
    preserve_index=False
)

dev_dataset = Dataset.from_pandas(
    dev_topic_df[["prompt"]],
    preserve_index=False
)

print(train_dataset)
print(dev_dataset)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)

In [ ]:
if 'tokenized_train' in globals():
    del tokenized_train

if 'tokenized_dev' in globals():
    del tokenized_dev

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["prompt"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["prompt"]
)

tokenized_dev = dev_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["prompt"]
)

print(tokenized_train.column_names)
print(type(tokenized_train[0]["input_ids"]))
print(len(tokenized_train[0]["input_ids"]))

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
test_batch = data_collator([
    tokenized_train[0],
    tokenized_train[1]
])

for key, value in test_batch.items():
    print(key, type(value), value.shape)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen_topic_lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator
)

print("Trainer ready.")

In [ ]:
test_batch = data_collator([
    tokenized_train[i]
    for i in range(2)
])

for key, value in test_batch.items():
    print(key, value.shape)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator
)

print("Trainer ready.")

In [ ]:
train_result = trainer.train()

In [ ]:
trainer.save_model("./qwen_topic_lora")
tokenizer.save_pretrained("./qwen_topic_lora")

print("LoRA model saved successfully.")

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",  # use your actual base model name
    torch_dtype="auto",
    device_map="auto"
)

model_lora = PeftModel.from_pretrained(base_model, "./qwen_topic_lora")
tokenizer = AutoTokenizer.from_pretrained("./qwen_topic_lora")

model_lora.eval()

In [ ]:
def generate_headline_finetuned(text):
    prompt = f"""You are a professional multilingual news editor.

Write a concise and informative news headline for the article below.

Important instructions:
- Write the headline in the same language as the article.
- Preserve the main event or information.
- Do not invent facts.
- Do not add explanations.
- Return ONLY the headline.
- Keep it concise.

Article:
{text}
"""

    messages = [{"role": "user", "content": prompt}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model_lora.device)

    with torch.no_grad():
        outputs = model_lora.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    headline = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    headline = re.sub(r"\s+", " ", headline).strip()

    return headline

In [ ]:
finetuned_headlines = []

for i in range(len(dev_data)):
    row = dev_data.iloc[i]
    headline = generate_headline_finetuned(row["text"])
    finetuned_headlines.append(headline)

    if i % 25 == 0:
        print(f"Processed {i}/{len(dev_data)}")

print("Done generating fine-tuned headlines.")

In [ ]:
# ROUGE-L
rouge_scores = []
for reference, prediction in zip(
    dev_data["headline"].fillna(""),
    finetuned_headlines
):
    score = rouge.score(str(reference), str(prediction))
    rouge_scores.append(score["rougeL"].fmeasure)

finetuned_rouge_l = np.mean(rouge_scores)
print(f"Fine-Tuned ROUGE-L: {finetuned_rouge_l:.6f}")

# BERTScore
from bert_score import score as bert_score_fn

P, R, F1 = bert_score_fn(
    finetuned_headlines,
    dev_data["headline"].fillna("").tolist(),
    lang="en",  # adjust if your dataset is multilingual and bertscore supports it
    verbose=False
)
finetuned_bertscore = F1.mean().item()
print(f"Fine-Tuned BERTScore: {finetuned_bertscore:.6f}")

# GenScore — use whatever formula you used for the other rows
# commonly something like an average of ROUGE-L and BERTScore
finetuned_genscore = (finetuned_rouge_l + finetuned_bertscore) / 2
print(f"Fine-Tuned GenScore: {finetuned_genscore:.6f}")

In [ ]:
%whos DataFrame

In [ ]:
new_row = pd.DataFrame([{
    "Approach": "Qwen2.5-0.5B LoRA Fine-Tuned",
    "Task": "Headline Generation",
    "Macro-F1": np.nan,
    "ROUGE-L": finetuned_rouge_l,
    "BERTScore": finetuned_bertscore,
    "GenScore": finetuned_genscore
}])

results = pd.concat([results, new_row], ignore_index=True)
results

In [ ]:
print(sample_submission.head(10))
print(sample_submission.columns.tolist())
print(test.head())
print(test.columns.tolist())

In [ ]:
%whos TfidfVectorizer
%whos LogisticRegression

In [ ]:
def classify_topic(text):
    X = tfidf.transform([text])
    pred = classifier.predict(X)[0]
    return pred

In [ ]:
# Topic classification on test set
test_topic_preds = []
for i in range(len(test)):
    row = test.iloc[i]
    pred = classify_topic(row["text"])  # TF-IDF+LogReg pipeline
    test_topic_preds.append(pred)
print("Topic classification done")    

# Headline generation on test set (LoRA fine-tuned)
test_headlines = []
for i in range(len(test)):
    row = test.iloc[i]
    headline = generate_headline_finetuned(row["text"])
    test_headlines.append(headline)
    if i % 50 ==0:
        print (f"Headline {i}/{len(test)}")
print("Headline generation done")

In [ ]:
submission_rows = []

for i in range(len(test)):
    row_id = test.iloc[i]["id"]
    
    submission_rows.append({
        "id": f"{row_id}_topic",
        "prediction": test_topic_preds[i]
    })
    
    submission_rows.append({
        "id": f"{row_id}_headline",
        "prediction": test_headlines[i]
    })

submission = pd.DataFrame(submission_rows)

print(len(submission))  # should be exactly 2 * len(test) = 3486
submission.head(10)

In [ ]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

In [ ]:
print(pd.Series(test_topic_preds).value_counts())

In [ ]:
print("Submission shape:", submission.shape)
print("\nColumns:")
print(submission.columns.tolist())

print("\nFirst 10 rows:")
display(submission.head(10))

print("\nLast 10 rows:")
display(submission.tail(10))

print("\nMissing predictions:")
print(submission["prediction"].isna().sum())

print("\nDuplicate IDs:")
print(submission["id"].duplicated().sum())

In [ ]:
topic_rows = submission[
    submission["id"].str.endswith("_topic")
]

headline_rows = submission[
    submission["id"].str.endswith("_headline")
]

print("Topic rows:", len(topic_rows))
print("Headline rows:", len(headline_rows))
print("Total rows:", len(submission))

In [ ]:
print("Validation results:")
print(results)

In [ ]:
print(type(model_lora))

In [ ]:
print("LoRA model is available.")
print("Device:", next(model_lora.parameters()).device)

In [ ]:
def lora_topic_prediction(text, language):
    prompt = f"""You are a multilingual news topic classifier.

The article is written in {language}.

Choose exactly ONE topic from:

business
health
politics
religion
sports
entertainment
technology

Article:
{text}

Topic:"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model_lora.device)

    with torch.no_grad():
        outputs = model_lora.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return extract_label(response)

In [ ]:
for i in range(5):
    row = dev_data.iloc[i]

    prediction = lora_topic_prediction(
        row["text"],
        row["lang"]
    )

    print("=" * 60)
    print("Language:", row["lang"])
    print("Actual:", row["category"])
    print("Predicted:", prediction)

In [ ]:
lora_final_score = (
    0.5 * lora_macro_f1 +
    0.5 * 0.502312
)

print(f"LoRA Final Score: {lora_final_score:.6f}")

In [ ]:
classical_final_score = (
    0.5 * classical_macro_f1 +
    0.5 * 0.477727
)

print(f"Classical Final Score: {classical_final_score:.6f}")

In [ ]:
zero_shot_final_score = (
    0.5 * 0.280338 +
    0.5 * 0.409144
)

print(f"Zero-Shot Qwen Final Score: {zero_shot_final_score:.6f}")

# DSN AI Bootcamp 2026 — Multilingual LLM Topic Classification & Headline Generation

## Project Overview

This project explores LLM-based approaches for multilingual news understanding and headline generation across four Nigerian languages: **Hausa, Igbo, Yoruba, and Nigerian Pidgin**.

The project compares classical machine learning, zero-shot prompting, few-shot prompting, and LoRA-based fine-tuning approaches for two related tasks:

1. **News topic classification**
2. **News headline generation**

The goal is to investigate how well a small multilingual language model performs compared with a traditional TF-IDF-based classifier and extractive headline generation baseline.

---

## Dataset

The dataset contains Nigerian news articles together with their associated language, category, headline, and other metadata.

The topic classification task contains seven categories:

- Business
- Health
- Politics
- Religion
- Sports
- Entertainment
- Technology

The available test set contains **1,743 articles**. The required submission contains two predictions for each article:

- Topic prediction
- Headline prediction

This results in **3,486 submission rows**.

Because the provided training data did not contain a separate development set, a **15% stratified validation split** was created from the training data using a fixed random seed.

---

## Methodology

The project evaluates the following approaches.

### 1. TF-IDF + Logistic Regression

A classical machine learning baseline was developed using:

- TF-IDF vectorization
- Unigrams and bigrams
- Logistic Regression
- Class balancing

This provided a strong reference point for the topic classification task.

### 2. Lead-1 Extractive Headline Generation

As a simple headline-generation baseline, the first sentence of each article was extracted and truncated to a maximum length.

This provides a reference point for evaluating generated headlines.

### 3. Qwen2.5-0.5B Zero-Shot

The multilingual **Qwen2.5-0.5B-Instruct** model was used without task-specific fine-tuning.

The model was prompted to classify articles into one of the seven predefined categories and was also prompted to generate headlines.

### 4. Qwen2.5-0.5B Few-Shot

Few-shot prompting was evaluated by providing the model with labelled examples before asking it to classify a new article.

Language-specific examples were used where available.

### 5. Qwen2.5-0.5B LoRA Fine-Tuning

Parameter-efficient fine-tuning was explored using LoRA.

The final LoRA configuration used:

- Rank (`r`): 8
- LoRA alpha: 16
- LoRA dropout: 0.05
- Target modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`

Only **1,081,344 parameters**, approximately **0.22%** of the model's 495 million parameters, were trainable.

---

## Evaluation Metrics

### Topic Classification

Topic classification was evaluated using **Macro-F1**, which gives equal importance to each topic category.

### Headline Generation

Headline generation was evaluated using:

- ROUGE-L F1
- BERTScore F1

The combined generation score was calculated as:

**GenScore = 0.5 × ROUGE-L + 0.5 × BERTScore**

The overall task score is defined as:

**Final Score = 0.5 × Macro-F1 + 0.5 × GenScore**

---

## Validation Results

| Approach | Task | Macro-F1 | ROUGE-L | BERTScore | GenScore |
|---|---|---:|---:|---:|---:|
| TF-IDF + Logistic Regression | Topic Classification | **0.835970** | — | — | — |
| Qwen2.5-0.5B Zero-Shot | Topic Classification | 0.280338 | — | — | — |
| Qwen2.5-0.5B Few-Shot | Topic Classification | 0.116628 | — | — | — |
| Lead-1 Extractive | Headline Generation | — | 0.238261 | 0.717193 | 0.477727 |
| Qwen2.5-0.5B Zero-Shot | Headline Generation | — | 0.160982 | 0.657305 | 0.409144 |
| Qwen2.5-0.5B LoRA Fine-Tuned | Headline Generation | — | 0.170768 | **0.833857** | **0.502312** |

---

## Results and Discussion

### Topic Classification

The classical TF-IDF + Logistic Regression model achieved a Macro-F1 of **0.835970** on the validation set.

The zero-shot Qwen2.5-0.5B approach achieved a Macro-F1 of **0.280338**, while the few-shot approach achieved **0.116628**.

These results show that, on this validation split, the classical supervised classifier was substantially more effective for topic classification than the prompting approaches tested.

The relatively low few-shot score also suggests that the particular examples and prompting strategy used were not sufficiently effective for this multilingual classification task.

### Headline Generation

The Lead-1 extractive baseline achieved a ROUGE-L score of **0.238261** and a GenScore of **0.477727**.

The zero-shot Qwen model achieved a ROUGE-L score of **0.160982** and a GenScore of **0.409144**.

After LoRA fine-tuning, the model achieved:

- ROUGE-L: **0.170768**
- BERTScore: **0.833857**
- GenScore: **0.502312**

The LoRA model produced a higher BERTScore and GenScore than both the extractive and zero-shot approaches, indicating improved semantic similarity between generated and reference headlines.

However, its ROUGE-L remained below the extractive baseline, showing that lexical overlap and semantic similarity can behave differently when evaluating generated headlines.

---

## Key Findings

1. **TF-IDF + Logistic Regression provided a strong baseline for topic classification**, achieving a Macro-F1 of 0.835970.

2. **Zero-shot prompting did not outperform the classical classifier** for topic classification.

3. **The few-shot prompting strategy performed worse than zero-shot prompting** on the validation set, indicating that prompt design and example selection are important factors in multilingual LLM classification.

4. **LoRA fine-tuning improved headline generation performance**, particularly in terms of BERTScore.

5. The LoRA headline-generation system achieved the highest observed **GenScore of 0.502312** among the tested headline-generation approaches.

6. The results demonstrate that a small LLM does not necessarily outperform a classical supervised model on every task. Model performance depends strongly on the task, language, training strategy, and evaluation metric.

---

## Submission

The final test predictions were generated for all **1,743 test articles**, producing the required **3,486 prediction rows**.

The submission was validated to ensure:

- No missing predictions
- No duplicate IDs
- Correct topic/headline ID structure
- Valid topic labels
- 1,743 topic predictions
- 1,743 headline predictions

The final submission file was saved as:

`/kaggle/working/submission.csv`

---

## Limitations

Several limitations should be considered when interpreting the results.

- The validation set was created from the available training data rather than using a predefined development set.
- The prompting approaches were evaluated using a small sub-1B model.
- Few-shot performance is sensitive to the selection and number of examples.
- The validation results may not perfectly represent performance on the hidden test set.
- The LoRA experiment reported here provides headline-generation results; a complete LoRA combined Final Score was not calculated because a corresponding LoRA topic-classification Macro-F1 was not established.

---

## Conclusion

This project compared classical machine learning and LLM-based approaches for multilingual Nigerian news topic classification and headline generation.

The experiments show a clear difference between the two tasks. Traditional TF-IDF + Logistic Regression was highly effective for supervised topic classification, while LoRA fine-tuning improved the semantic quality of generated headlines.

Overall, the project demonstrates the importance of evaluating both classical and LLM-based approaches rather than assuming that a larger or more sophisticated language model will automatically provide better performance across all tasks and languages.